# fetchr — Data Audit Notebook

**Purpose:** Before doing any machine learning, we need to understand what our data actually looks like. This notebook answers:
- How many dogs do we have?
- Which fields are missing data (nulls), and how often?
- What values do categorical fields like `size` and `breed` contain?
- How reliable are the boolean fields like `good_with_kids`?
- How useful is the free-text `description` field?

The answers tell us which fields we can trust as ML features and which ones need special handling.

## 1. Load the data

`pandas` is a library that loads tabular data into a structure called a **DataFrame** — essentially a programmable spreadsheet where each row is a dog and each column is a field.

`json_normalize` handles the fact that our data is a list of JSON objects — it flattens each object into a row.

In [7]:
import json
import pandas as pd

with open('fetchr.json') as f:
    raw = json.load(f)

df = pd.json_normalize(raw)

print(f'Rows (dogs): {len(df)}')
print(f'Columns (fields): {len(df.columns)}')
print(f'\nAll fields:\n{list(df.columns)}')

Rows (dogs): 222
Columns (fields): 31

All fields:
['id', 'source', 'source_id', 'source_url', 'name', 'breed_primary', 'breed_secondary', 'is_mixed', 'age_category', 'age_years_approx', 'size', 'gender', 'color', 'good_with_kids', 'good_with_dogs', 'good_with_cats', 'house_trained', 'special_needs', 'energy_level', 'shelter_name', 'city', 'state', 'zip', 'lat', 'lng', 'photos', 'description', 'tags', 'status', 'first_seen_at', 'last_updated_at']


## 2. Null audit — which fields are missing data?

A **null** means the scraper found no value for that field on PetFinder. 
This matters enormously for ML: if a field is null 80% of the time, it's not a reliable feature.

The table below shows, for each field:
- **null_count** — how many dogs are missing this value
- **null_pct** — what percentage of all dogs that is
- **filled_pct** — the inverse — how much of the field is actually usable

In [8]:
null_counts = df.isnull().sum()
null_pct = (null_counts / len(df) * 100).round(1)

null_audit = pd.DataFrame({
    'null_count': null_counts,
    'null_pct': null_pct,
    'filled_pct': (100 - null_pct),
}).sort_values('null_pct', ascending=False)

# Only show fields that have at least one null
null_audit[null_audit['null_count'] > 0]

,null_count,null_pct,filled_pct
age_years_approx,222,100.0,0.0
lat,222,100.0,0.0
lng,222,100.0,0.0
breed_secondary,185,83.3,16.7
good_with_cats,160,72.1,27.9
good_with_kids,94,42.3,57.7
good_with_dogs,61,27.5,72.5
color,45,20.3,79.7
house_trained,41,18.5,81.5
description,9,4.1,95.9


## 3. Categorical field distributions

For fields like `size`, `age_category`, `breed`, and `gender`, we want to know:
- What values actually appear in the data?
- Are they evenly spread or heavily skewed toward one value?

Just a list of column names you want to analyze. Defining it upfront means you can add/remove fields in one place instead of hunting through the code.

A heavily skewed field (e.g., 95% of dogs are "medium" size) is a weak ML feature — it doesn't help the model distinguish between dogs.

#### Understanding the Code:
```df[field].value_counts(dropna=False)
df[field] — selects one column (a Series)
.value_counts() — counts how many times each unique value appears, sorted descending automatically
dropna=False — includes nulls in the count rather than silently ignoring them
```

<mark>This is the critical flag. Without it, if 10 dogs have no size recorded, those rows disappear from your count and your percentages look cleaner than reality. dropna=False forces honesty.</mark>

In [15]:
categorical_fields = ['size', 'age_category', 'gender', 'status', 'energy_level']

for field in categorical_fields:
    print(f'\n--- {field} ---')
    counts = df[field].value_counts(dropna=False)
    pct = (counts / len(df) * 100).round(1)
    print(pd.DataFrame({'count': counts, 'pct': pct}).to_string())


--- size ---
        count   pct
size               
medium    139  62.6
large      54  24.3
small      28  12.6
xlarge      1   0.5

--- age_category ---
              count   pct
age_category             
adult            98  44.1
young            75  33.8
puppy            30  13.5
senior           19   8.6

--- gender ---
         count   pct
gender              
female     118  53.2
male       103  46.4
unknown      1   0.5

--- status ---
           count   pct
status                
available    220  99.1
adopted        2   0.9

--- energy_level ---
              count    pct
energy_level              
unknown         222  100.0


## Reading the Output

Size Distribution

| Value | Count | Pct |
|-------|-------|-----|
| medium | 139 | 62.6% ← dominant |
| large | 54 | 24.3% |
| small | 28 | 12.6% |
| xlarge | 1 | 0.5% ← almost never appears |

What This Tells You for ML

| Field | Verdict |
|-------|---------|
| `size` | **Weak** — 62.6% is `medium`, low discriminative power |
| `age_category` | **Decent** — reasonably spread across 4 values |
| `gender` | **Decent** — near 50/50 split, good balance |

## 4. Breed distribution

Breed is special — it likely has many unique values (high cardinality). 
<mark>High cardinality makes one-hot encoding impractical (you'd end up with hundreds of columns).</mark> One-hot encoding converts each unique value into its own column, filled with 0s and 1s.
This section shows the top breeds and how many unique breeds we have total.

In [16]:
print(f'Unique primary breeds: {df["breed_primary"].nunique()}')
print(f'\nTop 15 breeds:')
print(df['breed_primary'].value_counts().head(15).to_string())

Unique primary breeds: 31

Top 15 breeds:
breed_primary
Mixed Breed                       75
Pit Bull Terrier                  30
Bull Terrier                      14
Labrador Retriever                14
Siberian Husky                    14
Shepherd                          13
American Staffordshire Terrier     8
German Shepherd Dog                6
Canaan Dog                         5
Chihuahua                          5
Husky                              5
Black Labrador Retriever           3
Terrier                            3
Yorkshire Terrier                  3
Hound                              3


## 5. Boolean field distributions — true / false / unknown

The boolean fields (`good_with_kids`, `good_with_dogs`, `good_with_cats`, `house_trained`) are critical for matching — a user searching for a dog good with kids cares a lot about this.

<mark>But remember: **null ≠ false**. A null means PetFinder didn't specify. We need to see how many dogs have a known answer vs. unknown.</mark>

In [ ]:
boolean_fields = ['good_with_kids', 'good_with_dogs', 'good_with_cats', 'house_trained', 'special_needs', 'is_mixed']

for field in boolean_fields:
    counts = df[field].value_counts(dropna=False)
    pct = (counts / len(df) * 100).round(1)
    summary = pd.DataFrame({'count': counts, 'pct': pct})
    summary.index = summary.index.map(lambda x: 'unknown/null' if pd.isna(x) else ('yes' if x else 'no'))
    print(f'\n--- {field} ---')
    print(summary.to_string())

## 6. Tags — what personality traits appear and how often?

`tags` is a list field — each dog has zero or more tags like `"playful"`, `"calm"`, `"energetic"`.
This section unpacks all those lists, counts every unique tag, and shows the vocabulary.

This tells us: is the tag set small and normalized (good for ML), or a chaotic mix of freeform strings (needs cleaning)?

In [ ]:
from collections import Counter

# Flatten: each dog has a list of tags, we want one big list of all tags
all_tags = [tag for tags in df['tags'] if isinstance(tags, list) for tag in tags]

tag_counts = Counter(all_tags)
dogs_with_tags = df['tags'].apply(lambda t: isinstance(t, list) and len(t) > 0).sum()

print(f'Dogs with at least one tag: {dogs_with_tags} / {len(df)}')
print(f'Total unique tags: {len(tag_counts)}')
print(f'\nAll tags ranked by frequency:')
for tag, count in tag_counts.most_common():
    print(f'  {count:3d}x  {tag}')

## 7. Description audit — how useful is the free text?

`description` is the richest potential signal for semantic matching — it's the paragraph a shelter writes about the dog's personality. But it's only useful if:
- Enough dogs have one (coverage)
- They're long enough to contain real signal (length)

A 3-word description like "Sweet, gentle dog" is not very useful for embeddings. A 5-sentence paragraph is.

In [ ]:
has_description = df['description'].notna() & (df['description'].str.strip() != '')
desc_lengths = df.loc[has_description, 'description'].str.len()

print(f'Dogs with a description: {has_description.sum()} / {len(df)} ({has_description.mean()*100:.1f}%)')

if len(desc_lengths) > 0:
    print(f'\nDescription length (characters):')
    print(f'  shortest : {desc_lengths.min()}')
    print(f'  median   : {desc_lengths.median():.0f}')
    print(f'  longest  : {desc_lengths.max()}')
    print(f'\nSample description (first dog that has one):')
    sample = df.loc[has_description, 'description'].iloc[0]
    print(f'  "{sample[:300]}..."' if len(sample) > 300 else f'  "{sample}"')

## 8. Feature engineering readiness summary

Based on the audit above, this section scores each field by how ready it is for ML:
- **ready** — clean, low nulls, usable as-is after encoding
- **needs_handling** — has nulls or cardinality issues that need a decision
- **risky** — too sparse or inconsistent to rely on

In [ ]:
fields_to_assess = [
    'size', 'age_category', 'gender', 'breed_primary',
    'good_with_kids', 'good_with_dogs', 'good_with_cats',
    'house_trained', 'is_mixed', 'special_needs',
    'tags', 'description', 'color', 'energy_level',
]

print(f'{"Field":<20} {"Null%":>7}  {"Unique vals":>12}  Verdict')
print('-' * 65)

for field in fields_to_assess:
    if field not in df.columns:
        continue
    null_p = df[field].isnull().mean() * 100
    # For list fields, count dogs that have a non-empty list
    if df[field].apply(lambda x: isinstance(x, list)).any():
        unique = 'list field'
    else:
        unique = str(df[field].nunique())

    if null_p < 10:
        verdict = 'ready'
    elif null_p < 50:
        verdict = 'needs_handling'
    else:
        verdict = 'risky — too sparse'

    print(f'{field:<20} {null_p:>6.1f}%  {unique:>12}  {verdict}')